###  Create Bronze Delta Tables 

In [0]:
%python 
# Databricks Notebook: 02_create_bronze_delta_tables 
# Purpose: Write all banking datasets as Delta tables to Bronze container 
from pyspark.sql.functions import input_file_name, current_timestamp, lit 
from pyspark.sql.types import * 
from datetime import datetime 

print("="*60) 
print("CREATING BRONZE DELTA TABLES FOR ALL DATASETS") 
print("="*60) 
storage_account = "adlsbankinganalytics" 
raw_base_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/" 
bronze_base_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/" 


# ============================================ 
# Define schemas for all tables 
# ============================================ 
 
customers_schema = StructType([ 
    StructField("customer_id", StringType(), True), 
    StructField("first_name", StringType(), True), 
    StructField("last_name", StringType(), True), 
    StructField("dob", DateType(), True), 
    StructField("gender", StringType(), True), 
    StructField("phone", StringType(), True), 
    StructField("email", StringType(), True), 
    StructField("city", StringType(), True), 
    StructField("state", StringType(), True), 
    StructField("customer_since", DateType(), True), 
    StructField("kyc_status", StringType(), True), 
    StructField("occupation", StringType(), True), 
    StructField("annual_income", IntegerType(), True), 
    StructField("risk_category", StringType(), True) 
]) 
 
accounts_schema = StructType([ 
    StructField("account_id", StringType(), True), 
    StructField("customer_id", StringType(), True), 
    StructField("branch_id", StringType(), True), 
    StructField("account_type", StringType(), True), 
    StructField("account_number", StringType(), True), 
    StructField("open_date", DateType(), True), 
    StructField("balance", DoubleType(), True), 
    StructField("status", StringType(), True), 
    StructField("currency", StringType(), True), 
    StructField("nominee_registered", StringType(), True) 
]) 
 
transactions_schema = StructType([ 
    StructField("transaction_id", StringType(), True), 
    StructField("account_id", StringType(), True), 
    StructField("transaction_date", DateType(), True), 
    StructField("transaction_type", StringType(), True), 
    StructField("amount", DoubleType(), True), 
    StructField("merchant_name", StringType(), True), 
    StructField("payment_mode", StringType(), True), 
    StructField("status", StringType(), True), 
    StructField("city", StringType(), True), 
    StructField("channel", StringType(), True) 
]) 
 
loans_schema = StructType([ 
    StructField("loan_id", StringType(), True), 
    StructField("customer_id", StringType(), True), 
    StructField("loan_type", StringType(), True), 
    StructField("loan_amount", DoubleType(), True), 
    StructField("interest_rate", DoubleType(), True), 
    StructField("tenure_months", IntegerType(), True), 
    StructField("emi_amount", DoubleType(), True), 
    StructField("loan_start_date", DateType(), True), 
    StructField("loan_status", StringType(), True), 
    StructField("collateral_required", StringType(), True) 
]) 
 
credit_cards_schema = StructType([ 
    StructField("card_id", StringType(), True), 
    StructField("customer_id", StringType(), True), 
    StructField("card_type", StringType(), True), 
    StructField("credit_limit", IntegerType(), True), 
    StructField("available_limit", DoubleType(), True), 
    StructField("outstanding_balance", DoubleType(), True), 
    StructField("issue_date", DateType(), True), 
    StructField("expiry_date", DateType(), True), 
    StructField("card_status", StringType(), True) 
]) 
 
branches_schema = StructType([ 
    StructField("branch_id", StringType(), True), 
    StructField("branch_name", StringType(), True), 
    StructField("city", StringType(), True), 
    StructField("state", StringType(), True), 
    StructField("ifsc_code", StringType(), True), 
    StructField("manager_name", StringType(), True), 
    StructField("open_date", DateType(), True) 
]) 
 
employees_schema = StructType([ 
    StructField("employee_id", StringType(), True), 
    StructField("branch_id", StringType(), True), 
    StructField("employee_name", StringType(), True), 
    StructField("designation", StringType(), True), 
    StructField("salary", IntegerType(), True), 
    StructField("joining_date", DateType(), True), 
    StructField("employment_status", StringType(), True) 
]) 
 
fraud_schema = StructType([ 
    StructField("fraud_id", StringType(), True), 
    StructField("transaction_id", StringType(), True), 
    StructField("fraud_type", StringType(), True), 
    StructField("risk_score", IntegerType(), True), 
    StructField("detected_date", DateType(), True), 
    StructField("investigation_status", StringType(), True), 
    StructField("loss_amount", DoubleType(), True) 
]) 
 
insurance_schema = StructType([ 
    StructField("policy_id", StringType(), True), 
    StructField("customer_id", StringType(), True), 
    StructField("policy_type", StringType(), True), 
    StructField("premium_amount", IntegerType(), True), 
    StructField("sum_assured", IntegerType(), True), 
    StructField("start_date", DateType(), True), 
    StructField("end_date", DateType(), True), 
    StructField("status", StringType(), True) 
]) 
 
tickets_schema = StructType([ 
    StructField("ticket_id", StringType(), True), 
    StructField("customer_id", StringType(), True), 
    StructField("issue_type", StringType(), True), 
    StructField("priority", StringType(), True), 
    StructField("status", StringType(), True), 
    StructField("created_date", DateType(), True), 
    StructField("resolved_date", DateType(), True), 
    StructField("channel", StringType(), True) 
]) 


# ============================================ 
# Define schemas (same as above - omitted for brevity) 
# ============================================ 
# Read and Write each table as Delta 
# 1. Customers Table 
print("Processing Customers...") 
df_customers = spark.read.option("header", True).schema(customers_schema).csv(f"{raw_base_path}customers.csv") 
df_customers_with_metadata = df_customers \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("customers.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_customers_path = f"{bronze_base_path}customers/" 
df_customers_with_metadata.write.format("delta").mode("overwrite").save(bronze_customers_path) 
print(f"✓ Customers Delta table created") 

 
# 2. Accounts Table 
print("Processing Accounts...") 
df_accounts = spark.read.option("header", True).schema(accounts_schema).csv(f"{raw_base_path}accounts.csv") 
df_accounts_with_metadata = df_accounts \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("accounts.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_accounts_path = f"{bronze_base_path}accounts/" 
df_accounts_with_metadata.write.format("delta").mode("overwrite").save(bronze_accounts_path) 
print(f"✓ Accounts Delta table created") 
 

# 3. Transactions Table 
print("Processing Transactions...") 
df_transactions = spark.read.option("header", True).schema(transactions_schema).csv(f"{raw_base_path}transactions.csv") 
df_transactions_with_metadata = df_transactions \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("transactions.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_transactions_path = f"{bronze_base_path}transactions/" 
df_transactions_with_metadata.write.format("delta").mode("overwrite").save(bronze_transactions_path) 
print(f"✓ Transactions Delta table created") 
 

# 4. Loans Table 
print("Processing Loans...") 
df_loans = spark.read.option("header", True).schema(loans_schema).csv(f"{raw_base_path}loans.csv") 
df_loans_with_metadata = df_loans \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("loans.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_loans_path = f"{bronze_base_path}loans/" 
df_loans_with_metadata.write.format("delta").mode("overwrite").save(bronze_loans_path) 
print(f"✓ Loans Delta table created") 
 

# 5. Credit Cards Table 
print("Processing Credit Cards...") 
df_cards = spark.read.option("header", True).schema(credit_cards_schema).csv(f"{raw_base_path}credit_cards.csv") 
df_cards_with_metadata = df_cards \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("credit_cards.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_cards_path = f"{bronze_base_path}credit_cards/" 
df_cards_with_metadata.write.format("delta").mode("overwrite").save(bronze_cards_path) 
print(f"✓ Credit Cards Delta table created") 
 

# 6. Branches Table 
print("Processing Branches...") 
df_branches = spark.read.option("header", True).schema(branches_schema).csv(f"{raw_base_path}branches.csv") 
df_branches_with_metadata = df_branches \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("branches.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_branches_path = f"{bronze_base_path}branches/" 
df_branches_with_metadata.write.format("delta").mode("overwrite").save(bronze_branches_path) 
print(f"✓ Branches Delta table created") 
 

# 7. Employees Table 
print("Processing Employees...") 
df_employees = spark.read.option("header", True).schema(employees_schema).csv(f"{raw_base_path}employees.csv") 
df_employees_with_metadata = df_employees \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("employees.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_employees_path = f"{bronze_base_path}employees/" 
df_employees_with_metadata.write.format("delta").mode("overwrite").save(bronze_employees_path) 
print(f"✓ Employees Delta table created") 
 

# 8. Fraud Table 
print("Processing Fraud...") 
df_fraud = spark.read.option("header", True).schema(fraud_schema).csv(f"{raw_base_path}fraud_transactions.csv") 
df_fraud_with_metadata = df_fraud \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("fraud_transactions.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_fraud_path = f"{bronze_base_path}fraud_transactions/" 
df_fraud_with_metadata.write.format("delta").mode("overwrite").save(bronze_fraud_path) 
print(f"✓ Fraud Delta table created") 
 

# 9. Insurance Table 
print("Processing Insurance...") 
df_insurance = spark.read.option("header", True).schema(insurance_schema).csv(f"{raw_base_path}insurance_products.csv") 
df_insurance_with_metadata = df_insurance \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("insurance_products.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_insurance_path = f"{bronze_base_path}insurance_products/" 
df_insurance_with_metadata.write.format("delta").mode("overwrite").save(bronze_insurance_path) 
print(f"✓ Insurance Delta table created") 
 

# 10. Support Tickets Table 
print("Processing Support Tickets...") 
df_tickets = spark.read.option("header", 
True).schema(tickets_schema).csv(f"{raw_base_path}customer_support_tickets.csv") 
df_tickets_with_metadata = df_tickets \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit("customer_support_tickets.csv")) \
    .withColumn("ingestion_batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))) 
bronze_tickets_path = f"{bronze_base_path}customer_support_tickets/" 
df_tickets_with_metadata.write.format("delta").mode("overwrite").save(bronze_tickets_path) 
print(f"✓ Support Tickets Delta table created") 
 
print("\n✅ALL BRONZE DELTA TABLES CREATED SUCCESSFULLY!") 


### CREATE ALL BRONZE TABLES IN UNITY CATALOG

In [0]:
%sql

-- ============================================
-- CREATE ALL BRONZE TABLES IN UNITY CATALOG
-- ============================================

USE CATALOG banking_catalog;
USE SCHEMA bronze;

-- 1. Customers
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_customers
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/customers/';

-- 2. Accounts
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_accounts
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/accounts/';

-- 3. Transactions
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_transactions
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/transactions/';

-- 4. Loans
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_loans
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/loans/';

-- 5. Credit Cards
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_credit_cards
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/credit_cards/';

-- 6. Branches
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_branches
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/branches/';

-- 7. Employees
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_employees
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/employees/';

-- 8. Fraud Transactions
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_fraud_transactions
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/fraud_transactions/';

-- 9. Insurance Products
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_insurance_products
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/insurance_products/';

-- 10. Customer Support Tickets
CREATE TABLE IF NOT EXISTS banking_catalog.bronze.bronze_customer_support_tickets
USING DELTA
LOCATION 'abfss://bronze@adlsbankinganalytics.dfs.core.windows.net/customer_support_tickets/';

-- ============================================ 
-- VERIFY ALL TABLES CREATED 
-- ============================================ 
 -- Show all tables in bronze schema 
SHOW TABLES IN banking_catalog.bronze;



SELECT 'bronze_customers' AS table_name, COUNT(*) AS record_count
FROM banking_catalog.bronze.bronze_customers

UNION ALL

SELECT 'bronze_accounts', COUNT(*)
FROM banking_catalog.bronze.bronze_accounts

UNION ALL

SELECT 'bronze_transactions', COUNT(*)
FROM banking_catalog.bronze.bronze_transactions

UNION ALL

SELECT 'bronze_loans', COUNT(*)
FROM banking_catalog.bronze.bronze_loans

UNION ALL

SELECT 'bronze_credit_cards', COUNT(*)
FROM banking_catalog.bronze.bronze_credit_cards

UNION ALL

SELECT 'bronze_branches', COUNT(*)
FROM banking_catalog.bronze.bronze_branches

UNION ALL

SELECT 'bronze_employees', COUNT(*)
FROM banking_catalog.bronze.bronze_employees

UNION ALL

SELECT 'bronze_fraud_transactions', COUNT(*)
FROM banking_catalog.bronze.bronze_fraud_transactions

UNION ALL

SELECT 'bronze_insurance_products', COUNT(*)
FROM banking_catalog.bronze.bronze_insurance_products

UNION ALL

SELECT 'bronze_customer_support_tickets', COUNT(*)
FROM banking_catalog.bronze.bronze_customer_support_tickets;